[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Fluidos_y_Medios_Continuos_(FLD)/FLD-09_Fluidos_Turbulencia_Kolmogorov.ipynb)



# Investigación Avanzada: Fluidos Turbulencia Kolmogorov

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def simulate_2d_turbulence():
    # 2D pseudo-spectral turbulence simulation
    # Solves the 2D vorticity equation with forcing and hyperviscosity
    N = 256
    L = 2 * np.pi
    k = np.fft.fftfreq(N, L / (2 * np.pi * N))
    kx, ky = np.meshgrid(k, k)
    ksq = kx**2 + ky**2
    ksq[0, 0] = 1e-10  # Prevent division by zero
    
    # Hyperviscosity (to absorb energy at high k)
    nu = 1e-4
    hyper_order = 4
    dissipation = np.exp(-nu * (ksq**(hyper_order)) * 0.01) # integrating factor step

    # Initial vorticity in Fourier space
    omega_hat = np.zeros((N, N), dtype=complex)
    
    # Forcing band
    kf1, kf2 = 4.0, 6.0
    forcing_mask = (ksq >= kf1**2) & (ksq <= kf2**2)
    
    dt = 0.01
    steps = 1000
    
    for i in range(steps):
        # Compute velocity
        psi_hat = -omega_hat / ksq
        u_hat = 1j * ky * psi_hat
        v_hat = -1j * kx * psi_hat
        
        u = np.real(np.fft.ifft2(u_hat))
        v = np.real(np.fft.ifft2(v_hat))
        omega = np.real(np.fft.ifft2(omega_hat))
        
        # Compute nonlinear term: u * d(omega)/dx + v * d(omega)/dy
        domegadx_hat = 1j * kx * omega_hat
        domegady_hat = 1j * ky * omega_hat
        
        domegadx = np.real(np.fft.ifft2(domegadx_hat))
        domegady = np.real(np.fft.ifft2(domegady_hat))
        
        nonlinear = u * domegadx + v * domegady
        nonlinear_hat = np.fft.fft2(nonlinear)
        
        # Add random forcing at intermediate scales
        forcing_hat = (np.random.randn(N, N) + 1j * np.random.randn(N, N)) * forcing_mask
        forcing_hat *= 100.0 * np.sqrt(dt) # scale forcing
        
        # Time integration (Euler for simplicity in this demo)
        omega_hat = omega_hat - dt * nonlinear_hat + forcing_hat
        # Apply hyperviscosity
        omega_hat = omega_hat * dissipation
        
        omega_hat[0, 0] = 0.0 # zero mean vorticity
        
    # Calculate energy spectrum
    energy_hat = 0.5 * (np.abs(u_hat)**2 + np.abs(v_hat)**2)
    
    # Binning energy spectrum
    k_bins = np.arange(1, N//2)
    E_k = np.zeros(len(k_bins)-1)
    k_vals = np.sqrt(ksq)
    
    for i in range(len(k_bins)-1):
        mask = (k_vals >= k_bins[i]) & (k_vals < k_bins[i+1])
        E_k[i] = np.sum(energy_hat[mask])
        
    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    omega_final = np.real(np.fft.ifft2(omega_hat))
    im = ax1.imshow(omega_final, cmap='RdBu', extent=[0, L, 0, L])
    plt.colorbar(im, ax=ax1, label='Vorticity')
    ax1.set_title('2D Turbulence: Vorticity Field')
    
    # Plot spectrum
    kc = k_bins[:-1]
    ax2.loglog(kc, E_k, 'b-', label='Simulation')
    ax2.loglog(kc, 1e5 * kc**(-5/3), 'r--', label='k^{-5/3} (Kolmogorov)')
    ax2.loglog(kc, 1e7 * kc**(-3.0), 'g--', label='k^{-3} (Enstrophy cascade)')
    ax2.set_xlabel('Wavenumber k')
    ax2.set_ylabel('Energy Spectrum E(k)')
    ax2.set_title('Energy Spectrum')
    ax2.set_ylim(1e-5, np.max(E_k)*10)
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('Fluidos_Turbulencia_Kolmogorov.png')
    plt.show()

if __name__ == '__main__':
    simulate_2d_turbulence()
